# merge, join, Indicators, Validation, merge_asof, Reconciliation

## What you'll practice

`pd.merge` is SQL `JOIN`: `how=`, `on=`, `validate=` (catch unexpected duplicates!), `indicator=` (anti-joins), `suffixes=`. `merge_asof` does "latest value as of a time" — FX rates,
price lists, SCD lookups.

> Function and class names in **Implement** are what the tests look for. Run all tests for this notebook with `pytest tests/pandas/test_09_05_joins_merge.py -v` from the lab root. Untouched exercises show as *skipped*.
>
> Solutions: `solutions/09_pandas/05_joins_merge.ipynb`. Don't open it until you've tried.

### Setup — run this first (sample data and imports, not a solution)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
pd.set_option("display.width", 140)
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
DATA = LAB / "datasets"
orders_raw = pd.read_csv(DATA / "csv" / "orders.csv", dtype=str, keep_default_na=False)
print(orders_raw.shape); orders_raw.head()

## Exercise 1 — 🟢 Easy · Left join with validation

**Implement:** `with_customers(orders: pd.DataFrame, customers: pd.DataFrame) -> pd.DataFrame`

Left-join `customers[[customer_id, segment, country]]` onto orders by `customer_id`, with `validate="many_to_one"` and `suffixes=("", "_customer")`. Row count must equal the orders' count.

**Input:** `orders: pd.DataFrame, customers: pd.DataFrame`

**Expected output:** `pd.DataFrame`

**Constraints:** pandas / numpy; prefer vectorized column operations over row loops (`iterrows`, `apply(axis=1)`).

**Example:** the tests are executable examples — read `test_with_customers` in `tests/pandas/test_09_05_joins_merge.py`.

**Check:** `pytest tests/pandas/test_09_05_joins_merge.py -k test_with_customers`

**YOUR CODE ↓**

## Exercise 2 — 🟡 Medium · Anti-join with indicator

**Implement:** `orphans(orders: pd.DataFrame, customers: pd.DataFrame) -> pd.DataFrame`

Return orders whose `customer_id` is missing from customers (blank/NaN ids excluded), using `merge(..., indicator=True)` and `_merge == "left_only"`. Keep only the order columns.

**Input:** `orders: pd.DataFrame, customers: pd.DataFrame`

**Expected output:** `pd.DataFrame`

**Constraints:** pandas / numpy; prefer vectorized column operations over row loops (`iterrows`, `apply(axis=1)`).

**Example:** the tests are executable examples — read `test_orphans` in `tests/pandas/test_09_05_joins_merge.py`.

**Check:** `pytest tests/pandas/test_09_05_joins_merge.py -k test_orphans`

**YOUR CODE ↓**

## Exercise 3 — 🟡 Medium · Enrich with multiple dimensions

**Implement:** `enrich(orders, customers, products) -> pd.DataFrame`

Left-join customer `segment` and product `category`+`name` (rename to `product_name`). Blank/missing category → `"UNKNOWN"`. Validate both joins. Add `catalog_price` from products and `price_diff_pct` = (unit_price − catalog_price) / catalog_price × 100 rounded 1.

**Input:** `orders, customers, products`

**Expected output:** `pd.DataFrame`

**Constraints:** pandas / numpy; prefer vectorized column operations over row loops (`iterrows`, `apply(axis=1)`).

**Example:** the tests are executable examples — read `test_enrich` in `tests/pandas/test_09_05_joins_merge.py`.

**Check:** `pytest tests/pandas/test_09_05_joins_merge.py -k test_enrich`

**YOUR CODE ↓**

## Exercise 4 — 🟠 Hard · As-of joins for FX rates

**Implement:** `convert_currency(txns: pd.DataFrame, rates: pd.DataFrame) -> pd.DataFrame`

`txns`: `txn_id, ts (datetime), currency, amount`; `rates`: `currency, effective_ts, rate_to_inr` (unsorted). Attach the latest rate with `effective_ts <= ts` for the same currency via
`pd.merge_asof(..., by="currency", direction="backward")`. INR always has rate 1.0. `amount_inr` rounded 2. Return in the original txn order with the original index.
Txns before any rate get NaN.

**Input:** `txns: pd.DataFrame, rates: pd.DataFrame`

**Expected output:** `pd.DataFrame`

**Constraints:** pandas / numpy; prefer vectorized column operations over row loops (`iterrows`, `apply(axis=1)`).

**Example:** the tests are executable examples — read `test_convert_currency` in `tests/pandas/test_09_05_joins_merge.py`.

**Check:** `pytest tests/pandas/test_09_05_joins_merge.py -k test_convert_currency`

**YOUR CODE ↓**

## Challenge — 🔴 Very Hard · Reconcile two datasets · 🧩 Mini-project

**Implement:** `reconcile_frames(source: pd.DataFrame, target: pd.DataFrame, key: str, columns: list[str], tolerance: float = 0.0) -> tuple[dict, pd.DataFrame]`

Outer-merge on `key` with `indicator=True` and suffixes `_src`/`_tgt`. Return `(summary, mismatches)`:
`summary = {"source_rows", "target_rows", "only_in_source", "only_in_target", "in_both", "rows_with_mismatch"}` (ints);
`mismatches` = long-format DataFrame with columns `key, column, source, target` for each differing value among keys present in both (numeric: `abs diff > tolerance`; otherwise
`!=` with both-NaN counted as equal), sorted by key then column order. Duplicate keys on either side → `ValueError` (check with `validate="one_to_one"`).

**Input:** `source: pd.DataFrame, target: pd.DataFrame, key: str, columns: list[str], tolerance: float = 0.0`

**Expected output:** `tuple[dict, pd.DataFrame]`

**Constraints:** pandas / numpy; prefer vectorized column operations over row loops (`iterrows`, `apply(axis=1)`).

**Example:** the tests are executable examples — read `test_reconcile_frames, test_reconcile_frames_duplicates` in `tests/pandas/test_09_05_joins_merge.py`.

**Check:** `pytest tests/pandas/test_09_05_joins_merge.py -k "test_reconcile_frames or test_reconcile_frames_duplicates"`

**YOUR CODE ↓**